In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
# 데이터 로드 및 기본 정보
# 원본 데이터 로드
df_original = pd.read_csv('df_merged.csv')
df = df_original.copy()  # 전처리용 복사본

In [20]:
# 타겟 변수 정의
target_columns = ['Leaf_TPC', 'Root_TPC', 'Leaf_TFC', 'Root_TFC']

### 전처리 결과 및 분석

In [2]:
# 데이터 로드
X = pd.read_csv('X_preprocessed.csv')
y = pd.read_csv('y_targets.csv')
scenario_info = pd.read_csv('scenario_info.csv')

In [5]:
X.shape
y.shape
scenario_info.shape

(405, 1)

### True/False, 0/1 분석

In [7]:
# 데이터 타입 확인
X.dtypes.value_counts()

float64    30
bool        8
Name: count, dtype: int64

In [8]:
# Boolean 타입 컬럼 확인
boolean_columns = X.select_dtypes(include=['bool']).columns.tolist()
print(f"\nBoolean 타입 컬럼 ({len(boolean_columns)}개):")
for col in boolean_columns:
    print(f"  - {col}")


Boolean 타입 컬럼 (8개):
  - scenario_SSP1
  - scenario_SSP3
  - scenario_SSP5
  - stage_reproductive
  - stage_senescence
  - stage_vegetative_early
  - stage_vegetative_late
  - stage_vegetative_mid


In [9]:
# 0/1 형태의 numeric 컬럼 확인
numeric_binary_columns = []
for col in X.select_dtypes(include=[np.number]).columns:
    unique_vals = X[col].unique()
    # 스케일링 전 원래 0/1이었을 가능성이 있는 컬럼
    if col.endswith('stress') or col.endswith('optimal') or \
       col.endswith('toxic') or col.endswith('anomaly') or \
       col.endswith('transition') or col.endswith('violation'):
        numeric_binary_columns.append(col)

print(f"\n0/1 Binary 지표 컬럼 ({len(numeric_binary_columns)}개):")
for col in numeric_binary_columns:
    # 스케일링된 값의 범위 확인
    print(f"  - {col}: [{X[col].min():.3f}, {X[col].max():.3f}]")


0/1 Binary 지표 컬럼 (8개):
  - energy_violation: [0.000, 0.000]
  - vpd_stress: [-0.787, 1.270]
  - temp_optimal: [-1.096, 0.912]
  - heat_stress: [-0.912, 1.096]
  - cold_stress: [0.000, 0.000]
  - co2_toxic: [-1.399, 0.715]
  - ssp3_june_anomaly: [-0.283, 3.536]
  - september_transition: [-0.512, 1.955]


### StandardScaler 영향 분석

In [10]:
# 주요 수치형 변수들의 스케일링 상태 확인
numeric_features = ['month', 'CO2ppm', 'Temp', 'Humid', 'VPD', 
                   'Chl_a', 'Chl_b', 'Car', 'PI_abs', 'Fv-Fm']

In [11]:
# 주요 변수들 스케일링된 값 범위
for col in numeric_features[:5]:
    if col in X.columns:
        mean_val = X[col].mean()
        std_val = X[col].std()
        min_val = X[col].min()
        max_val = X[col].max()
        print(f"{col:15}: 평균={mean_val:6.3f}, 표준편차={std_val:6.3f}, "
              f"범위=[{min_val:6.2f}, {max_val:6.2f}]")

month          : 평균=-0.000, 표준편차= 1.001, 범위=[ -1.69,   1.39]
CO2ppm         : 평균= 0.000, 표준편차= 1.001, 범위=[ -1.58,   1.26]
Temp           : 평균= 0.000, 표준편차= 1.001, 범위=[ -2.85,   1.39]
Humid          : 평균= 0.000, 표준편차= 1.001, 범위=[ -2.75,   3.40]
VPD            : 평균= 0.000, 표준편차= 1.001, 범위=[ -2.64,   1.50]


### 문제점
- True/False와 0/1 혼재
- Binary 변수 스케일링 -> 0/1이어야 할 변수들 음수나 양수가 됨
- 시나리오 정보 분리되어 있음

In [16]:
# Boolean을 0/1로 변환
X_fixed = X.copy()
for col in boolean_columns:
    X_fixed[col] = X_fixed[col].astype(int)

print("변환 예시:")
print(f"  변환 전 (Boolean): {X[boolean_columns[0]].iloc[0]}")
print(f"  변환 후 (0/1): {X_fixed[boolean_columns[0]].iloc[0]}")

변환 예시:
  변환 전 (Boolean): True
  변환 후 (0/1): 1


In [17]:
# 선택적 스케일링 예시
from sklearn.preprocessing import StandardScaler

# 원본 데이터 다시 로드 (스케일링 전)
df_original = pd.read_csv('df_merged.csv')

# 연속형 변수만 선택
continuous_features = ['month', 'CO2ppm', 'Temp', 'Humid', 'VPD',
                      'Chl_a', 'Chl_b', 'TChl', 'Car', 'Chl_a_b', 'TCh-Car',
                      'ABS-RC', 'Dio-RC', 'Tro-RC', 'Eto-RC',
                      'PI_abs', 'DF_abs', 'SFI_abs', 'Fv-Fm']

# Binary 변수 (스케일링 제외)
binary_features = ['extreme_stress_fv', 'vpd_stress', 'temp_optimal',
                  'heat_stress', 'cold_stress', 'co2_toxic',
                  'energy_violation', 'ssp3_june_anomaly', 'september_transition']

print("\n연속형 변수 (스케일링 대상):", len(continuous_features))
print("Binary 변수 (스케일링 제외):", len(binary_features))

print("""
### 개선방안 3: 스케일링 방법 선택
""")

print("\n각 모델별 권장 스케일링:")
recommendations = {
    "Tree 기반 모델 (RandomForest, XGBoost)": "스케일링 불필요",
    "선형 모델 (LinearRegression, Ridge, Lasso)": "StandardScaler 권장",
    "신경망 (Neural Network)": "MinMaxScaler 권장",
    "SVM": "StandardScaler 필수",
    "KNN": "StandardScaler 또는 MinMaxScaler 필수"
}

for model, scaling in recommendations.items():
    print(f"  • {model}: {scaling}")


연속형 변수 (스케일링 대상): 19
Binary 변수 (스케일링 제외): 9

### 개선방안 3: 스케일링 방법 선택


각 모델별 권장 스케일링:
  • Tree 기반 모델 (RandomForest, XGBoost): 스케일링 불필요
  • 선형 모델 (LinearRegression, Ridge, Lasso): StandardScaler 권장
  • 신경망 (Neural Network): MinMaxScaler 권장
  • SVM: StandardScaler 필수
  • KNN: StandardScaler 또는 MinMaxScaler 필수


### 수정된 전처리

In [18]:
def improved_preprocessing(df, target_columns, scale_method='selective'):
    """
    개선된 전처리 함수
    
    Parameters:
    -----------
    df: 원본 데이터프레임
    target_columns: 타겟 변수 리스트
    scale_method: 'none', 'all', 'selective'
    """
    
    # 타겟 분리
    X = df.drop(columns=target_columns + ['Leaf_ExtractionYield', 'Root_ExtractionYield'])
    y = df[target_columns]
    
    # 시나리오 정보 저장
    scenario = X['scenario'].copy()
    
    # 원핫인코딩 (dtype=int로 통일)
    X = pd.get_dummies(X, columns=['scenario'], prefix='scenario', dtype=int)
    
    # Feature Engineering (기존 코드 활용)
    # 4-1. 생장 단계 변수
    growth_stage_map = {
        5: 'vegetative_early',
        6: 'vegetative_mid',
        7: 'vegetative_late',
        8: 'reproductive',
        9: 'senescence'
    }
    X['growth_stage'] = X['month'].map(growth_stage_map)
    X = pd.get_dummies(X, columns=['growth_stage'], prefix='stage')

    # 4-2. 스트레스 지표
    X['vpd_stress'] = (X['VPD'] > 2.5).astype(int)
    print(f"VPD > 2.5 kPa: {X['vpd_stress'].sum()}개 ({X['vpd_stress'].mean()*100:.1f}%)")
    
    # 온도 스트레스 (15-25°C 적정)
    X['temp_optimal'] = ((X['Temp'] >= 15) & (X['Temp'] <= 25)).astype(int)
    X['heat_stress'] = (X['Temp'] > 25).astype(int)
    X['cold_stress'] = (X['Temp'] < 15).astype(int)

    # CO2 독성 (800ppm 이상)
    X['co2_toxic'] = (X['CO2ppm'] > 800).astype(int)
    print(f"CO2 > 800ppm (독성): {X['co2_toxic'].sum()}개")

    # 4-3. 복합 스트레스 지수
    # 온도-CO2 복합 스트레스
    X['stress_index'] = (X['Temp'] - 20) * (X['CO2ppm'] - 432) / 432

    # SSP3 6월 이상 플래그
    X['ssp3_june_anomaly'] = ((X['scenario_SSP3'] == 1) & (X['month'] == 6)).astype(int)
    print(f"SSP3 6월 이상: {X['ssp3_june_anomaly'].sum()}개")

    # 4-4. 광합성 효율 지표
    # 에너지 효율 (전자전달/포획)
    X['energy_efficiency'] = np.where(X['Tro-RC'] > 0, 
                                    X['Eto-RC'] / X['Tro-RC'], 
                                    0)

    # 열소산 비율 (보호 메커니즘)
    X['dissipation_ratio'] = np.where(X['ABS-RC'] > 0,
                                        X['Dio-RC'] / X['ABS-RC'],
                                        0)

    # 4-5. 9월 생리적 전환 플래그 (이상치 x)
    X['september_transition'] = (X['month'] == 9).astype(int)
    print(f"9월 데이터: {X['september_transition'].sum()}개 (자료조사: 자원 재분배 시기)")

    # 스케일링 전략
    if scale_method == 'none':
        print("스케일링 없음 (Tree 모델용)")
        X_scaled = X
        
    elif scale_method == 'all':
        print("전체 스케일링 (선형 모델용)")
        scaler = StandardScaler()
        X_scaled = pd.DataFrame(
            scaler.fit_transform(X),
            columns=X.columns,
            index=X.index
        )
        
    elif scale_method == 'selective':
        print("선택적 스케일링 (권장)")
        X_scaled = X.copy()
        
        # 연속형 변수만 스케일링
        continuous_cols = []
        binary_cols = []
        categorical_cols = []
        
        for col in X.columns:
            # 원핫인코딩된 변수
            if col.startswith('scenario_') or col.startswith('stage_'):
                categorical_cols.append(col)
            # Binary 지표
            elif 'stress' in col or 'optimal' in col or 'toxic' in col or \
                 'anomaly' in col or 'transition' in col or 'violation' in col:
                binary_cols.append(col)
            # 연속형 변수
            else:
                continuous_cols.append(col)
        
        # 연속형만 스케일링
        if continuous_cols:
            scaler = StandardScaler()
            X_scaled[continuous_cols] = scaler.fit_transform(X[continuous_cols])
        
        print(f"  - 연속형 (스케일링): {len(continuous_cols)}개")
        print(f"  - Binary (유지): {len(binary_cols)}개") 
        print(f"  - Categorical (유지): {len(categorical_cols)}개")
    
    return X_scaled, y, scenario

In [21]:
print("1. Tree 모델 (XGBoost, RandomForest):")
X_tree, y, scenario = improved_preprocessing(df, target_columns, 'none')

1. Tree 모델 (XGBoost, RandomForest):
VPD > 2.5 kPa: 155개 (38.3%)
CO2 > 800ppm (독성): 268개
SSP3 6월 이상: 30개
9월 데이터: 84개 (자료조사: 자원 재분배 시기)
스케일링 없음 (Tree 모델용)


In [22]:
print("\n2. 선형 모델 (Lasso, Ridge):")
X_linear, y, scenario = improved_preprocessing(df, target_columns, 'selective')


2. 선형 모델 (Lasso, Ridge):
VPD > 2.5 kPa: 155개 (38.3%)
CO2 > 800ppm (독성): 268개
SSP3 6월 이상: 30개
9월 데이터: 84개 (자료조사: 자원 재분배 시기)
선택적 스케일링 (권장)
  - 연속형 (스케일링): 21개
  - Binary (유지): 8개
  - Categorical (유지): 8개


In [23]:
print("\n3. 신경망 모델:")
X_nn, y, scenario = improved_preprocessing(df, target_columns, 'selective') # 추가로 MinMaxScaler 적용 권장


3. 신경망 모델:
VPD > 2.5 kPa: 155개 (38.3%)
CO2 > 800ppm (독성): 268개
SSP3 6월 이상: 30개
9월 데이터: 84개 (자료조사: 자원 재분배 시기)
선택적 스케일링 (권장)
  - 연속형 (스케일링): 21개
  - Binary (유지): 8개
  - Categorical (유지): 8개


In [24]:
# 전처리된 데이터 저장
X_tree.to_csv('X_preprocessed_tree.csv', index=False)
X_linear.to_csv('X_preprocessed_linear.csv', index=False)
X_nn.to_csv('X_preprocessed_nn.csv', index=False)
y.to_csv('y_targets.csv', index=False)